# TFM - Predicción de cancelaciones
## 1. Predicción de Cancelaciones de Reservas de Hotel

**Autor:** Grupo 4  
**Fecha:** 2026-02-18

---

### MODELADO:
- Generar modelo de predicción

In [ ]:
!pip install lightgbm catboost category_encoders optuna

# **1) Inicio**

---



In [ ]:
import pandas as pd
import numpy as np
import os
import csv
import json
import shap
import textwrap
import warnings
from datetime import datetime

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

# Scikit-learn: Preprocesamiento y Selección de Modelos
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance

# Scikit-learn: Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn: Métricas
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay,
    RocCurveDisplay, precision_recall_curve
)

# Otros Modelos y Encoders
from category_encoders import TargetEncoder
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import optuna
import joblib
# Configuración
pd.set_option('display.float_format', lambda x: '%.5f' % x)

mi_marron = LinearSegmentedColormap.from_list("mi_marron", ["#f8f9fa", "#8b733a"])

color_maestro = '#6e550c'
plt.rcParams.update({
    'text.color': color_maestro,          # Color de todos los textos
    'axes.labelcolor': color_maestro,     # Color de etiquetas X e Y
    'axes.edgecolor': color_maestro,      # Color de los bordes (ejes)
    'xtick.color': color_maestro,         # Color de los números en X
    'ytick.color': color_maestro,         # Color de los números en Y
    'axes.titlecolor': color_maestro      # Color de los títulos
})

# Hora y fecha
hora_actual = datetime.now().strftime("%H:%M:%S")
fecha_folder = datetime.now().strftime("%Y-%m-%d")

# Rutas
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

ruta_base = "/content/drive/MyDrive/TFM_Aurea_Team/02_notebooks/Modelado/metricas_ejecuciones/"
ruta_dia = os.path.join(ruta_base, fecha_folder)
ruta_final = os.path.join(ruta_dia, hora_actual)
os.makedirs(ruta_final, exist_ok=True)


warnings.filterwarnings('ignore')


In [ ]:
# Ruta al archivo CSV generado en el EDA
ruta_datos = '/content/drive/MyDrive/TFM_Aurea_Team/01_data/processed/df_unificado_completo_v8.csv'

# Cargar el dataset
df = pd.read_csv(ruta_datos)

print('Datos cargados correctamente')
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')

### - Metricas:

In [ ]:
metricas = ['Modelo',
            'auc_train',
            'auc_test',
            'precision_recall_train',
            'precision_recall_test',
            'accuracy_train',
            'accuracy_test',
            'recall_train',
            'recall_test',
            'precision_train',
            'precision_test',
            'f1_score_train',
            'f1_score_test',
            'sobreajuste',
            'variables_utilizadas'
            ]


# **Función entrenar y evaluar**

In [ ]:

def entrenar_evaluar(evaluacion,variables, nombre_modelo, modelo, X_t, X_tes, y_t, y_tes, X_val, y_val):

  metricas2 = ['Modelo',
            'AUC',

            'Precision-Recall',

            'Accuracy',
            'Recall',
            'Precision',
            'Sobreajuste'
            ]

  evaluacion2 = pd.DataFrame(columns=metricas2)

  # Entrenamos el modelo:
  if  'LightGBM' in nombre_modelo:

    modelo.fit(X_t, y_t,
               eval_set=[(X_val, y_val)]
               )


  elif 'CatBoost' in nombre_modelo:

    modelo.fit(X_t, y_t,
               eval_set=[(X_val, y_val)],
               verbose=100 # Imprime cada 100 iteraciones para no saturar la pantalla
              )

  elif 'Random Forest' in nombre_modelo:

    modelo.fit(X_t, y_t)


  else:

    modelo.fit(X_t, y_t,
              eval_set=[(X_val, y_val)],
              verbose=50

              )

  # Predecimos:
  pred_train = modelo.predict(X_t)
  pred_test = modelo.predict(X_tes)
  prob_train = modelo.predict_proba(X_t)[:, 1]
  prob_test = modelo.predict_proba(X_tes)[:, 1]


  #----------------------------------------
  #.  PRUEBA:
  #-------------------------------------------

  #umbral_seguro = 0.40
  #probabilidades_train = modelo.predict_proba(X_t)[:, 1]
  #probabilidades_test = modelo.predict_proba(X_tes)[:, 1]
  #pred_train = (probabilidades_train >= umbral_seguro).astype(int)
  #pred_test = (probabilidades_test >= umbral_seguro).astype(int)

  # Métricas Train
  auc_train = roc_auc_score(y_t, prob_train)
  pres_recall_train = average_precision_score(y_t, prob_train)
  acc_train = accuracy_score(y_t, pred_train)
  recall_train = recall_score(y_t, pred_train)
  precision_train = precision_score(y_t, pred_train)
  f1_score_train = f1_score(y_t, pred_train)

  # Métricas Test
  auc_test = roc_auc_score(y_tes, prob_test)
  pres_recall_test = average_precision_score(y_tes, prob_test)
  acc_test = accuracy_score(y_tes, pred_test)
  recall_test = recall_score(y_tes, pred_test)
  precision_test = precision_score(y_tes, pred_test)
  f1_score_test = f1_score(y_tes, pred_test)

  # Sobreajuste
  sobreajuste = pres_recall_train - pres_recall_test

  #-------------------------------------------------------

  # Probado bajar el umbral y subimos mucho el recall de clase 1. Es decir, de los clientes que realmente cancelaron
  # detectamos un 17 % más pero nos baja mucho la precision (Los que realmente cancelan de los que el modelo predice)
  hora_actual = datetime.now().strftime("%H:%M:%S")
  nombre_con_hora = f"{nombre_modelo} ({hora_actual})"

  evaluacion.loc[len(evaluacion)] = {'Modelo': nombre_con_hora,
                                     'auc_train': round(auc_train,3),
                                     'auc_test': round(auc_test,3),
                                     'precision_recall_train' : round(pres_recall_train,3),
                                     'precision_recall_test' :  round(pres_recall_test,3),
                                     'accuracy_train' : round(acc_train,3),
                                     'accuracy_test' : round(acc_test,3),
                                     'recall_train' : round(recall_train,3),
                                     'recall_test' : round(recall_test,3),
                                     'precision_train' : round(precision_train,3),
                                     'precision_test': round(precision_test,3),
                                     'f1_score_train': round(f1_score_train,3),
                                     'f1_score_test': round(f1_score_test,3),
                                     'sobreajuste': round(sobreajuste,3),
                                     'variables_utilizadas': variables
                                    }

  evaluacion2.loc[len(evaluacion2)] = {'Modelo': nombre_modelo,
                                     'AUC': round(auc_test,2),

                                     'Precision-Recall' :  round(pres_recall_test,2),

                                     'Accuracy' : round(acc_test,2),

                                     'Recall' : round(recall_test,2),
                                     'precision_train' : round(precision_train,2),
                                     'Precision': round(precision_test,2),
                                     'Sobreajuste': round(sobreajuste,3)
                                    }

  print(evaluacion2.iloc[len(evaluacion2)-1])
  print()
  print(evaluacion.iloc[len(evaluacion)-1])
  print()
  print(classification_report(y_tes, pred_test))
  print()

  #---------
  #GRAFICOS
  #---------
  fig, ax = plt.subplots(2, 2, figsize=(15, 12))
  plt.subplots_adjust(hspace=0.3, wspace=0.3)

  # --- Subplot 1: Matriz de Confusión ---
  cm = confusion_matrix(y_tes, pred_test)
  disp_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['CANCELADOS', 'NO CANCELADOS'])
  disp_cm.plot(ax=ax[0, 0], cmap=mi_marron, values_format='d')
  ax[0, 0].set_title('Matriz de Confusión')

  # --- Subplot 2: Curva Precision-Recall ---

  ap_score = average_precision_score(y_tes, prob_test)
  PrecisionRecallDisplay.from_predictions(y_tes, prob_test, ax=ax[0, 1], color="#8b733a")
  ax[0, 1].set_title(f'Curva Precision-Recall (AP = {ap_score:.3f})')

  # --- Subplot 3: Curva ROC ---

  roc_visualizacion = RocCurveDisplay.from_predictions(y_tes, prob_test, ax=ax[1, 0], color="#8b733a")
  roc_auc = roc_visualizacion.roc_auc
  ax[1, 0].plot([0, 1], [0, 1], color='orange', lw=2, linestyle='--') # Línea aleatoria
  ax[1, 0].set_title(f'Curva ROC (AUC = {roc_auc:.3f})')

  # --- Subplot 4: Resumen de Métricas ---

  stats_text = (
      f"Métricas:\n"
      f"Accuracy: {acc_test:.3f}\n"
      f"AUC: {auc_test:.3f}\n"
      f"Recall (Detección): {recall_test:.3f}\n"
      f"Precision (Seguridad): {precision_test:.3f}\n"
      f"F1-Score: {f1_score_test:.3f}\n"
      f"Sobreajuste (en precision - recall): {sobreajuste:.3f}\n"
  )
  ax[1, 1].text(0.5, 0.5, stats_text, fontsize=14, ha='center', va='center',
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
  ax[1, 1].axis('off')


  plt.tight_layout()

  plt.show()

  return modelo, evaluacion

# **Funciónes encoder_train_test**

In [ ]:
# Funcion que realiza one hot encoder o target encoder en funcion de la cardinalidad (para XGBoost)
# Hace train_test_split

def encoder_train_test():

  global X_train_D,  X_test_D,  y_train_D,  y_test_D
  global X_train_t_D, X_val_D, y_train_t_D, y_val_D


  cols_categoricas = df_modelado.select_dtypes(include=['object']).columns

  alta_cardinalidad = [col for col in cols_categoricas if df_modelado[col].nunique() > 10]
  baja_cardinalidad = [col for col in cols_categoricas if df_modelado[col].nunique() <= 10]

  # Aplicamos One-Hot Encoding para categoricas con baja cardinalidad
  if baja_cardinalidad:
      print(f"Aplicando One-Hot Encoding a columnas con baja cardinalidad: {baja_cardinalidad}")
      df_modelado_dum = pd.get_dummies(df_modelado, columns=baja_cardinalidad,
                                     drop_first = True,
                                     dtype=int)

  cols_categoricas = df_modelado_dum.select_dtypes(include=['object']).columns
  alta_cardinalidad = [col for col in cols_categoricas if df_modelado_dum[col].nunique() > 10]


  # Dividimos el DataSet en train, test
  X_train_D,  X_test_D,  y_train_D,  y_test_D = train_test_split(df_modelado_dum.drop('CANCELADO',axis=1),
                                                                 df_modelado_dum['CANCELADO'],
                                                                 test_size=0.30,
                                                                 stratify=df_modelado_dum['CANCELADO'],
                                                                 random_state=1041
                                                                 )


  # Apicamos Target Encoder con la categoriacas con alta cardinalidad
  if alta_cardinalidad:
      print(f"Aplicando One-Hot Encoding a columnas con baja cardinalidad: {alta_cardinalidad}")
      encoder = TargetEncoder(cols=alta_cardinalidad)
      X_train_D = encoder.fit_transform(X_train_D,
                                        y_train_D)
      X_test_D = encoder.transform(X_test_D)


  X_train_t_D, X_val_D, y_train_t_D, y_val_D = train_test_split(X_train_D,
                                                                y_train_D,
                                                                test_size=0.15,
                                                                random_state=42,
                                                                stratify=y_train_D
                                                                )
  print()
  print(f"Split finalizado:")
  print(f"   - Train principal: {X_train_t_D.shape[0]} filas")
  print(f"   - Validación:      {X_val_D.shape[0]} filas")
  print(f"   - Test (Oculto):   {X_test_D.shape[0]} filas")
  print()


# ----------------------------------------------------------
# FUNCION train_test_split() para catboost y ligthGBM
# ----------------------------------------------------------


def train_test_val():
  global X_train, X_test, y_train, y_test
  global X_train_t, X_val, y_train_t, y_val

  X_train, X_test, y_train, y_test = train_test_split(df_modelado.drop('CANCELADO',axis=1),
                                                      df_modelado['CANCELADO'],
                                                      test_size=0.30,
                                                      stratify=df_modelado['CANCELADO'],
                                                      random_state=42
                                                      )

  # Separamos un set de Validación del Train .
  X_train_t, X_val, y_train_t, y_val = train_test_split(X_train,
                                                        y_train,
                                                        test_size=0.15,
                                                        random_state=42,
                                                        stratify=y_train
                                                        )
  print()
  print(f"Split finalizado:")
  print(f"   - Train principal: {X_train_t.shape[0]} filas")
  print(f"   - Validación:      {X_val.shape[0]} filas")
  print(f"   - Test (Oculto):   {X_test.shape[0]} filas")
  print()

# **Funciones Importancia Variables**

In [ ]:


def importancia_variables(modelo, x_test, Y_test):

  print("Calculando Permutation Importance ...")

  result = permutation_importance(modelo,
                                  x_test,
                                  Y_test,
                                  n_repeats=10,
                                  random_state=42,
                                  n_jobs=-1,
                                  scoring='f1' # O 'auc'
                                  )

  df_perm = pd.DataFrame({'Variable': x_test.columns,
                          'Importancia_Media': result.importances_mean,
                          'Desviacion': result.importances_std
                         }).sort_values(by='Importancia_Media', ascending=False)

  df_importantes = df_perm.sort_values(by='Importancia_Media', ascending=False).reset_index(drop=True)

  variables = df_importantes[round(df_importantes['Importancia_Media'],3) > 0]['Variable'].tolist()
  variables.append('CANCELADO')

  #  Graficamos
  plt.figure(figsize=(10, 8))
  sns.barplot(data=df_perm.head(20), x='Importancia_Media', y='Variable', hue='Variable', palette='magma', legend=False)
  plt.xlabel('Importancia')
  plt.title('Permutation Importance (Impacto en f1)')
  plt.show()
  return df_importantes, variables

In [ ]:
def funcion_shap(modelo, X_test):


  for col in X_test.select_dtypes(['category']).columns:

        col_idx = modelo.feature_name().index(col)
        X_test[col] = X_test[col].cat.set_categories(modelo.pandas_categorical[col_idx])

  # Creamos el "Explainer"
  explainer = shap.TreeExplainer(modelo)

  # Calculamos el objeto Explanation directamente
  explanation = explainer(X_test)

  # Nota: En clasificación binaria, si 'explanation' tiene 3 dimensiones (muestras, variables, clases),
  # seleccionamos la clase 1 (CANCELADOS).
  if len(explanation.shape) == 3:
      explanation_to_plot = explanation[:, :, 1]
  else:
      explanation_to_plot = explanation

  # --- Visualización ---
  plt.figure(figsize=(10, 8))
  shap.plots.beeswarm(explanation_to_plot, max_display=14)

  # álculo de importancia
  # Los valores SHAP están en .values
  feature_importance_shap = np.abs(explanation_to_plot.values).mean(0)

  shap_df = pd.DataFrame({
      'variable': X_test.columns,
      'importancia_shap': feature_importance_shap
  })
  shap_df = shap_df.sort_values(by='importancia_shap', ascending=False)

  # Seleccionamos las top 14 variables
  variables = shap_df['variable'].head(14).tolist()
  print(f"Variables seleccionadas por SHAP: {variables}")

  # Añadimos el target para el análisis posterior
  variables.append('CANCELADO')

  return shap_df, variables

In [ ]:
def funcion_shap(modelo, X_test):
  # Shap Value:
  import shap
  import numpy as np
  import pandas as pd

  # Creamos el "Explainer" específico para modelos basados en árboles (TreeExplainer)
  # Es extremadamente rápido con LightGBM
  explainer = shap.TreeExplainer(modelo)

  # Calculamos los valores SHAP para el conjunto de test
  # Esto calcula la contribución de cada variable para cada reserva individual
  shap_values = explainer.shap_values(X_test)

  # Nota: En problemas binarios, shap_values suele devolver una lista de dos matrices.
  # Usamos la segunda [1] que corresponde a la probabilidad de CANCELAR (Clase 1).
  if isinstance(shap_values, list):
      shap_values_to_plot = shap_values[1]
  else:
      shap_values_to_plot = shap_values

  # Grafico:
  plt.figure(figsize=(10, 8))
  shap.summary_plot(shap_values, X_test, plot_type="dot")

  # Calculamos la importancia media (magnitud del impacto)
  feature_importance_shap = np.abs(shap_values).mean(0)
  shap_df = pd.DataFrame({'variable': X_test.columns, 'importancia_shap': feature_importance_shap})
  shap_df = shap_df.sort_values(by='importancia_shap', ascending=False)

  # Seleccionamos, por ejemplo, las top 14 variables
  variables = shap_df['variable'].head(14).tolist()
  print(f"Variables seleccionadas por SHAP: {variables}")
  variables.append('CANCELADO')

  return shap_df, variables

# **Funcion Instanciar modelos**

In [ ]:
def instanciar(eval,nombre_modelo, lista_variables):

  global X_train_t, X_test, X_val

  if  'LightGBM' in nombre_modelo:

    variables_categoricas = df_modelado.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    for col in variables_categoricas:
      X_train_t[col] = X_train[col].astype('category')
      X_test[col] = X_test[col].astype('category')
      X_val[col] = X_val[col].astype('category')

    modelo = LGBMClassifier(objective='binary',
                            metric='auc',
                            n_estimators=100,
                            learning_rate=0.1,
                            random_state=42,
                            n_jobs=-1,
                            is_unbalance=True,
                            verbose=0
                            )


    modelo, eval = entrenar_evaluar(eval,
                                    lista_variables,
                                    nombre_modelo,
                                    modelo,
                                    X_train_t,
                                    X_test,
                                    y_train_t,
                                    y_test,
                                    X_val,
                                    y_val
                                    )

  elif 'CatBoost' in nombre_modelo:

    variables_categoricas = df_modelado.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    modelo = CatBoostClassifier(iterations=200,
                                learning_rate=0.1,
                                depth=6,
                                cat_features=variables_categoricas,
                                random_state=42
                                )

    modelo, eval = entrenar_evaluar(eval,
                                    lista_variables,
                                    nombre_modelo,
                                    modelo,
                                    X_train_t,
                                    X_test,
                                    y_train_t,
                                    y_test,
                                    X_val,
                                    y_val
                                   )


  elif 'Random Forest' in nombre_modelo:

    modelo= RandomForestClassifier(n_estimators=100,
                                   criterion='gini',
                                   max_depth=None,
                                   min_samples_split=2,
                                   class_weight='balanced',
                                   n_jobs=-1,
                                   random_state=42
                                   )

    modelo, eval = entrenar_evaluar(eval,
                                    lista_variables,
                                    nombre_modelo,
                                    modelo,
                                    X_train_t_D,
                                    X_test_D,
                                    y_train_t_D,
                                    y_test_D,
                                    X_val_D,
                                    y_val_D
                                    )

  else:

    modelo = xgb.XGBClassifier(objective='binary:logistic',
                               n_estimators=100,
                               scale_pos_weight=1.67,
                               learning_rate=0.1,
                               n_jobs=-1,
                               random_state=42,
                               eval_metric='logloss',
                              )






    modelo, eval = entrenar_evaluar(eval,
                                    lista_variables,
                                    nombre_modelo,
                                    modelo,
                                    X_train_t_D,
                                    X_test_D,
                                    y_train_t_D,
                                    y_test_D,
                                    X_val_D,
                                    y_val_D
                                    )

  return modelo, eval

# **Funcion correlacion**

In [ ]:
def correlacion():
 corr = df_modelado.select_dtypes(include='number').corr()
 plt.figure(figsize = (20,10))
 sns.heatmap(corr, annot=True, cmap="coolwarm")
 plt.title('Correlaciones')
 plt.show()

# **Funcion rendimiento**

In [ ]:

def rendimiento(proba_prediccion, prediccion, realidad, ruta_carpeta=''):
    # 1. Configuración de estilo y tiempo
    plt.style.use('seaborn-v0_8-white')


    # --- GRÁFICA 1: CURVAS (ROC Y PRECISION-RECALL) ---
    fig1, ax1 = plt.subplots(1, 2, figsize=(15, 6), facecolor='white')

    # Curva Precision-Recall
    ap_score = average_precision_score(realidad, proba_prediccion)
    PrecisionRecallDisplay.from_predictions(realidad, proba_prediccion, ax=ax1[0], color="#005b96")
    ax1[0].set_title(f'Curva Precision-Recall\n(AP = {ap_score:.3f})', fontweight='bold')
    ax1[0].set_facecolor('#f0f8ff')
    ax1[0].patch.set_facecolor('#f0f8ff')

    # Curva ROC
    roc_visualizacion = RocCurveDisplay.from_predictions(realidad, proba_prediccion, ax=ax1[1], color="#005b96")
    roc_auc = roc_visualizacion.roc_auc
    ax1[1].plot([0, 1], [0, 1], color='orange', lw=2, linestyle='--')
    ax1[1].set_title(f'Curva ROC\n(AUC = {roc_auc:.3f})', fontweight='bold')
    ax1[1].set_facecolor('#f0f8ff')
    ax1[1].patch.set_facecolor('#f0f8ff')

    # Formato de bordes negros para Gráfica 1
    for ax in ax1:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor('black')
            spine.set_linewidth(1.5)

    nombre_curvas = f"auc_precision_recall_{hora_actual}.png"
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_carpeta, nombre_curvas), dpi=300, facecolor=fig1.get_facecolor())
    plt.show()

    # --- GRÁFICA 2: MATRIZ DE CONFUSIÓN ---
    fig2, ax2 = plt.subplots(1, 1, figsize=(8, 6), facecolor='white')

    cm = confusion_matrix(realidad, prediccion)
    disp_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NO CANCELADOS', 'CANCELADOS'])
    disp_cm.plot(ax=ax2, cmap='Blues', values_format='d')
    ax2.set_title('Matriz de Confusión', fontweight='bold', pad=20)

    # Formato de bordes negros para Gráfica 2
    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

    nombre_matriz = f"matriz_confusion_{hora_actual}.png"
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_carpeta, nombre_matriz), dpi=300, facecolor=fig2.get_facecolor())
    plt.show()

    print(f"Gráficas guardadas:\n1. {nombre_curvas}\n2. {nombre_matriz}")

# **Funcion comprativa_roc**

In [ ]:

def comparativa_roc(modelos):

  color_m = '#8b733a'
  paleta = sns.color_palette("YlOrBr", len(modelos))
  paleta = ['#DD8452', '#C44E52', '#937860']
  fig, ax = plt.subplots(1, 2, figsize=(16, 6))

  for i, (nombre, modelo) in enumerate(modelos.items()):
      if (nombre == 'LightGBM') or (nombre == 'CatBoost'):
        # ROC en el primer recuadro
        RocCurveDisplay.from_estimator(modelo,
                                       X_test,
                                       y_test,
                                       ax=ax[0],
                                       name=nombre,
                                       color=paleta[i]
                                       )
        # PR en el segundo recuadro
        PrecisionRecallDisplay.from_estimator(modelo,
                                              X_test,
                                              y_test,
                                              ax=ax[1],
                                              name=nombre,
                                              color=paleta[i]
                                              )

      else:
        # ROC en el primer recuadro
        RocCurveDisplay.from_estimator(modelo,
                                       X_test_D,
                                       y_test,
                                       ax=ax[0],
                                       name=nombre,
                                       color=paleta[i]
                                       )
        # PR en el segundo recuadro
        PrecisionRecallDisplay.from_estimator(modelo,
                                              X_test_D,
                                              y_test,
                                              ax=ax[1],
                                              name=nombre,
                                              color=paleta[i]
                                              )


  # Personalización de ambos
  for a in ax:
      a.tick_params(colors=color_m)
      a.xaxis.label.set_color(color_m)
      a.yaxis.label.set_color(color_m)
      a.title.set_color(color_m)
      a.grid(True, linestyle='--', alpha=0.4)

  ax[0].set_title('Curvas ROC')
  ax[1].set_title('Curvas Precision-Recall')

  plt.tight_layout()
  plt.savefig('comparativa_curvas.png', dpi=300, bbox_inches='tight')
  plt.show()

# **2) Con variables originales**

In [ ]:
def lista_variables():
  variables =  ['LEAD_TIME',
              'CANCELADO',
              'NOCHES',
              'ADULTOS', 'NENES', 'BEBES',
              'SUPLETORIA', 'CUNAS',
              'MONEDA',
              'COMERCIALIZADORA',
        #     'CANAL_CONSOLIDADO', mismo resultado sin canal_consolidado y con fuente_negocio,_egmento_cliente, y mejor que canal_consolidado sol
              'GRATIS',

        #     'NOMBRE_HOTEL'
        #     'HOTEL_COMPLEJO',        ----- funcionan mejor las: 'HOTEL_HABITACION_TOP','COMPLEJO_RESERVA', 'COMPLEJO_RESERVA_TOP',
        #     'NOMBRE_HABITACION',
              'Categoria',

              'TIENE_FIDELIDAD', 'LOYALTY_REWARDS',
        #     'PAIS_TOP', # No funciona, mejor PAIS
        #     'PAIS',
              'TIPO_VIAJERO',
              'ES_GRUPO',
              'ADR',
              'VALOR_RESERVA', # ---- probados las tres en todas la combinaciones y mejor juntos que por separado o ninguno. POr separado sobreajusta un poco menos (1%)
              'REV_PAX',

        #      'DIA_SEMANA_LLEGADA_SIN',
        #      'DIA_SEMANA_LLEGADA_COS', # mejoran juntos
              'MES_LLEGADA_SIN',
              'MES_LLEGADA_COS',
        #     'FIN_DE_SEMANA_INCLUIDO',   ------- probada y no es representativa
        #      'FIN_DE_SEMANA_LLEGADA',
              'HORA_TOMA_SIN',
              'HORA_TOMA_COS',
        #     'HORA_MINU_TOMA_SIN',
        #     'HORA_MINU_TOMA_COS',    ------ Probadas sin mejoras

         #    'HABITACION_TOP',
              'HOTEL_HABITACION_TOP', # ----- pruebo esta
              'COMPLEJO_RESERVA',           # funciona mejor con hotel_habitacion y complejoreserva top y no top
         #     'COMPLEJO_RESERVA_TOP',

         #    'TEMPORADA_ALTA', - probada y no cambia nada
         #    'MES_LLEGADA'
         #    'DIA_LLEGADA',
         #    'DIA_SEMANA_FECHA_TOMA',
         #    'DIA_SEMANA_LLEGADA',
         #    'DIA_LLEGADA_SIN',
         #    'DIA_LLEGADA_COS',


         #    'DIA_SEMANA_FECHA_TOMA_SIN',
         #    'DIA_SEMANA_FECHA_TOMA_COS',  #  ----- estas 4 no suman juntas, ni por pares
         #    'MES_FECHA_TOMA_SIN',
         #    'MES_FECHA_TOMA_COS',

         #    'DIA_SEMANA_FECHA_TOMA_COS',
         #    'DIA_AÑO_LLEGADA_SIN',
         #    'DIA_AÑO_LLEGADA_COS',     ------ Probadas y no mejoran
         #    'SEMANA_AÑO_LLEGADA_SIN',
         #    'SEMANA_AÑO_LLEGADA_COS',
         #    'NOCHES_CAT', # - Probada junto a sin y cos, y sola y no mejora


         #   'TIPO_NORMALIZADO', -- Repetida
         #   'TIPO_CLIENTE', ---- Probado. No mejora

              'PAIS_TOP_200', # - probada y no mejora.Dejor por si es mas eficiente
         #    'PAIS_TOP_100', # - probada y no mejora
         #    'PAIS_TOP_60', # - Probada y no
         #    'PAIS_RESIDENCIA',
         #    'DISTANCIA_PAIS', # Probada y nada
         #    'FIN_DE_SEMANA_LLEGADA',  # probada y no mejora
         #    'FIN_DE_SEMANA_INCLUIDO',

         #    'FUENTE_NEGOCIO', 'SEGMENTO',  'CLIENTE' # Probadas juntas y por separados y en combinaciones. Mejor 3 juntasa

               'FUENTE_NEGOCIO_SEGMENTO_CLIENTE' # NUeva a probar. Mejora que con respecto a 'FUENTE_NEGOCIO', 'SEGMENTO',
    #          'FUENTE_NEGOCIO_SEGMENTO_CLIENTE_TOP_MEDIANA'


      #        'CLIENTE_TOP_200',
      #        'CLIENTE_TOP',
      #        'CLIENTE_TOP_10',
      #        'CLIENTE_TOP_50',
      #        'CLIENTE_TOP_100'

     #         'CLIENTE_TOP_MEDIANA'
     #         'CLIENTE_TOP_MEDIANA_X2'
     #         'CLIENTE_TOP_MEDIANA_X3'


     #         'FUENTE_NEGOCIO_SEGMENTO',
           #   'FUENTE_NEGOCIO_CLIENTE', --------------mejor FUENTE_NEGOCIO_SEGMENTO_CLIENTE que la tres por separado
           #   'SEGMENTO_CLIENTE'
           #   'CLIENTE' - sin efecto en las metricas
           #   'MES_SALIDA', 'DIA_SALIDA', 'DIA_SEMANA_SALIDA', 'SEASON_SALIDA',
           #   'MES_FECHA_TOMA', 'DIA_FECHA_TOMA', 'DIA_SEMANA_FECHA_TOMA', 'SEASON_FECHA_TOMA',
           #   'MES_FECHA_MOD', 'DIA_FECHA_MOD', 'DIA_SEMANA_FECHA_MOD', 'SEASON_FECHA_MOD'
            ]
  return variables

In [ ]:
evaluacion = pd.DataFrame(columns=metricas)

In [ ]:
variables =  ['LEAD_TIME',
              'CANCELADO',
              'NOCHES',
              'ADULTOS', 'NENES', 'BEBES',
              'SUPLETORIA', 'CUNAS',
              'MONEDA',
              'COMERCIALIZADORA',
        #     'CANAL_CONSOLIDADO', mismo resultado sin canal_consolidado y con fuente_negocio,_egmento_cliente, y mejor que canal_consolidado sol
              'GRATIS',

        #     'NOMBRE_HOTEL'
        #     'HOTEL_COMPLEJO',        ----- funcionan mejor las: 'HOTEL_HABITACION_TOP','COMPLEJO_RESERVA', 'COMPLEJO_RESERVA_TOP',
        #     'NOMBRE_HABITACION',
              'Categoria',

              'TIENE_FIDELIDAD', 'LOYALTY_REWARDS',
        #     'PAIS_TOP', # No funciona, mejor PAIS
        #     'PAIS',
              'TIPO_VIAJERO',
              'ES_GRUPO',
              'ADR',
              'VALOR_RESERVA', # ---- probados las tres en todas la combinaciones y mejor juntos que por separado o ninguno. POr separado sobreajusta un poco menos (1%)
              'REV_PAX',

        #      'DIA_SEMANA_LLEGADA_SIN',
        #      'DIA_SEMANA_LLEGADA_COS', # mejoran juntos
              'MES_LLEGADA_SIN',
              'MES_LLEGADA_COS',
        #     'FIN_DE_SEMANA_INCLUIDO',   ------- probada y no es representativa
        #      'FIN_DE_SEMANA_LLEGADA',
              'HORA_TOMA_SIN',
              'HORA_TOMA_COS',
        #     'HORA_MINU_TOMA_SIN',
        #     'HORA_MINU_TOMA_COS',    ------ Probadas sin mejoras

         #    'HABITACION_TOP',
              'HOTEL_HABITACION_TOP', # ----- pruebo esta
              'COMPLEJO_RESERVA',           # funciona mejor con hotel_habitacion y complejoreserva top y no top
         #     'COMPLEJO_RESERVA_TOP',

         #    'TEMPORADA_ALTA', - probada y no cambia nada
         #    'MES_LLEGADA'
         #    'DIA_LLEGADA',
         #    'DIA_SEMANA_FECHA_TOMA',
         #    'DIA_SEMANA_LLEGADA',
         #    'DIA_LLEGADA_SIN',
         #    'DIA_LLEGADA_COS',


         #    'DIA_SEMANA_FECHA_TOMA_SIN',
         #    'DIA_SEMANA_FECHA_TOMA_COS',  #  ----- estas 4 no suman juntas, ni por pares
         #    'MES_FECHA_TOMA_SIN',
         #    'MES_FECHA_TOMA_COS',

         #    'DIA_SEMANA_FECHA_TOMA_COS',
         #    'DIA_AÑO_LLEGADA_SIN',
         #    'DIA_AÑO_LLEGADA_COS',     ------ Probadas y no mejoran
         #    'SEMANA_AÑO_LLEGADA_SIN',
         #    'SEMANA_AÑO_LLEGADA_COS',
         #    'NOCHES_CAT', # - Probada junto a sin y cos, y sola y no mejora


         #   'TIPO_NORMALIZADO', -- Repetida
         #   'TIPO_CLIENTE', ---- Probado. No mejora

              'PAIS_TOP_200', # - probada y no mejora.Dejor por si es mas eficiente
         #    'PAIS_TOP_100', # - probada y no mejora
         #    'PAIS_TOP_60', # - Probada y no
         #    'PAIS_RESIDENCIA',
         #    'DISTANCIA_PAIS', # Probada y nada
         #    'FIN_DE_SEMANA_LLEGADA',  # probada y no mejora
         #    'FIN_DE_SEMANA_INCLUIDO',

         #    'FUENTE_NEGOCIO', 'SEGMENTO',  'CLIENTE' # Probadas juntas y por separados y en combinaciones. Mejor 3 juntasa

               'FUENTE_NEGOCIO_SEGMENTO_CLIENTE' # NUeva a probar. Mejora que con respecto a 'FUENTE_NEGOCIO', 'SEGMENTO',
    #          'FUENTE_NEGOCIO_SEGMENTO_CLIENTE_TOP_MEDIANA'


      #        'CLIENTE_TOP_200',
      #        'CLIENTE_TOP',
      #        'CLIENTE_TOP_10',
      #        'CLIENTE_TOP_50',
      #        'CLIENTE_TOP_100'

     #         'CLIENTE_TOP_MEDIANA'
     #         'CLIENTE_TOP_MEDIANA_X2'
     #         'CLIENTE_TOP_MEDIANA_X3'


     #         'FUENTE_NEGOCIO_SEGMENTO',
           #   'FUENTE_NEGOCIO_CLIENTE', --------------mejor FUENTE_NEGOCIO_SEGMENTO_CLIENTE que la tres por separado
           #   'SEGMENTO_CLIENTE'
           #   'CLIENTE' - sin efecto en las metricas
           #   'MES_SALIDA', 'DIA_SALIDA', 'DIA_SEMANA_SALIDA', 'SEASON_SALIDA',
           #   'MES_FECHA_TOMA', 'DIA_FECHA_TOMA', 'DIA_SEMANA_FECHA_TOMA', 'SEASON_FECHA_TOMA',
           #   'MES_FECHA_MOD', 'DIA_FECHA_MOD', 'DIA_SEMANA_FECHA_MOD', 'SEASON_FECHA_MOD'
            ]

In [ ]:

# Preparamos los datos
df_modelado = df[variables].copy()
encoder_train_test()
train_test_val()

## **2.1) Random Forest**

In [ ]:
rf, evaluacion = instanciar(evaluacion,
                              'Random Forest',
                              variables
                              )

## **2.1) XGBoost**

In [ ]:

xg, evaluacion = instanciar(evaluacion,
                              'XGBoost',
                              variables
                              )

## **2.2) CatBoost**

In [ ]:
catB, evaluacion = instanciar(evaluacion,
                              'CatBoost',
                               variables
                              )

## **2.3) LightGBM**

In [ ]:

lightB, evaluacion = instanciar(evaluacion,
                              'LightGBM',
                               variables
                              )

## **2.4) Comparativa Curvas ROC**





In [ ]:
modelos = {
      'LightGBM': lightB,
      'XGBoost': xg,
      'CatBoost': catB
  }

comparativa_roc(modelos)

### - Importacia de las variables:

**a) Mediante Shap Values**

In [ ]:
df_importantes, variables = funcion_shap(lightB, X_test)

if (('HORA_TOMA_SIN' not in variables) & ('HORA_TOMA_COS' in variables)):
  variables.append('HORA_TOMA_SIN')


df_modelado = df[variables].copy()
train_test_val()
encoder_train_test()

**b) Mediante Permutation Importance**

In [ ]:
# IMPORTANCIA DE VARIABLES:
# ---------------------------------------
df_importantes, variables = importancia_variables(lightB, X_test, y_test)

if (('HORA_TOMA_SIN' not in variables) & ('HORA_TOMA_COS' in variables)):
  variables.append('HORA_TOMA_SIN')

df_modelado = df[variables].copy()
train_test_val()
encoder_train_test()


In [ ]:

lightB, evaluacion = instanciar(evaluacion,
                              'LightGBM  Variables permutation',
                              variables
                              )


# **3) Con nuevas variables**

In [ ]:
variables = ['NOCHES', 'REGIMEN', 'ADULTOS', 'NENES', 'BEBES',
       'TIPO_CLIENTE', 'MONEDA',
       'SUPLETORIA', 'CUNAS',
       'COMERCIALIZADORA',
       'GRATIS', 'NOMBRE_HOTEL', 'Categoria', 'VALOR_RESERVA', 'TIPO_VIAJERO', 'LEAD_TIME',
       'CANCELADO', 'TIENE_FIDELIDAD','FUENTE_NEGOCIO', 'SEGMENTO',  'CLIENTE', 'ES_GRUPO'
       ]

In [ ]:
variables = ['LEAD_TIME',
             'CANCELADO',
             'NOCHES',
             'ADULTOS', 'NENES', 'BEBES',
             'SUPLETORIA', 'CUNAS',
             'MONEDA',
             'COMERCIALIZADORA',

             'GRATIS',

         #   'NOMBRE_HOTEL',
         #   'HOTEL_COMPLEJO',        ----- funcionan mejor las: 'HOTEL_HABITACION_TOP','COMPLEJO_RESERVA', 'COMPLEJO_RESERVA_TOP',
         #   'NOMBRE_HABITACION',
             'Categoria',

             'TIENE_FIDELIDAD', 'LOYALTY_REWARDS',
         #   'PAIS_TOP', # No funciona, mejor PAIS
         #   'PAIS',
             'TIPO_VIAJERO',
             'ES_GRUPO',

             'ADR',
             'VALOR_RESERVA', # ---- probados las tres en todas la combinaciones y mejor juntos que por separado o ninguno. POr separado sobreajusta un poco menos (1%)
             'REV_PAX',

             'DIA_SEMANA_LLEGADA_SIN',
             'DIA_SEMANA_LLEGADA_COS', # mejoran juntos
             'MES_LLEGADA_SIN',
             'MES_LLEGADA_COS',
        #     'FIN_DE_SEMANA_INCLUIDO',   ------- probada y no es representativa
             'FIN_DE_SEMANA_LLEGADA',
             'HORA_TOMA_SIN',
             'HORA_TOMA_COS',
        #    'HORA_MINU_TOMA_SIN',
        #    'HORA_MINU_TOMA_COS',    ------ Probadas sin mejoras

        #    'HABITACION_TOP',
             'HOTEL_HABITACION_TOP',
             'COMPLEJO_RESERVA',           # funciona mejor con hotel_habitacion y complejoreserva top y no top
             'COMPLEJO_RESERVA_TOP',

        #    'TEMPORADA_ALTA', - probada y no cambia nada
        #    'MES_LLEGADA',
        #    'DIA_LLEGADA',
        #    'DIA_SEMANA_FECHA_TOMA',
        #    'DIA_SEMANA_LLEGADA',
        #    'DIA_LLEGADA_SIN',
        #    'DIA_LLEGADA_COS',


        #    'DIA_SEMANA_FECHA_TOMA_SIN',
        #    'DIA_SEMANA_FECHA_TOMA_COS',  #  ----- estas 4 no suman juntas, ni por pares
        #    'MES_FECHA_TOMA_SIN',
        #    'MES_FECHA_TOMA_COS',

        #    'DIA_SEMANA_FECHA_TOMA_COS',
        #    'DIA_AÑO_LLEGADA_SIN',
        #    'DIA_AÑO_LLEGADA_COS',     ------ Probadas y no mejoran
        #    'SEMANA_AÑO_LLEGADA_SIN',
        #    'SEMANA_AÑO_LLEGADA_COS',
        #    'NOCHES_CAT', # - Probada junto a sin y cos, y sola y no mejora


        #   'TIPO_NORMALIZADO', -- Repetida
        #   'TIPO_CLIENTE', ---- Probado. No mejora

            'PAIS_TOP_200', # - probada y no mejora.Dejor por si es mas eficiente
       #    'PAIS_TOP_100', # - probada y no mejora
       #    'PAIS_TOP_60', # - Probada y no
       #    'PAIS_RESIDENCIA',
       #    'DISTANCIA_PAIS', # Probada y nada
       #    'FIN_DE_SEMANA_LLEGADA',  # probada y no mejora
       #    'FIN_DE_SEMANA_INCLUIDO',

        #   'FUENTE_NEGOCIO', 'SEGMENTO',  'CLIENTE' # Probadas juntas y por separados y en combinaciones. Mejor 3 juntasa

        #  'FUENTE_NEGOCIO_SEGMENTO_CLIENTE' # NUeva a probar. Mejora que con respecto a 'FUENTE_NEGOCIO', 'SEGMENTO',  'CLIENTE'

           'CLIENTE_TOP_200'
  #        'CLIENTE_TOP',
  #        'CLIENTE_TOP_10',
  #        'CLIENTE_TOP_50',
  #        'CLIENTE_TOP_100',


        #   'FUENTE_NEGOCIO_SEGMENTO',
        #   'FUENTE_NEGOCIO_CLIENTE', --------------mejor FUENTE_NEGOCIO_SEGMENTO_CLIENTE que la tres por separado
        #   'SEGMENTO_CLIENTE'
        #   'CLIENTE' - sin efecto en las metricas
        #   'MES_SALIDA', 'DIA_SALIDA', 'DIA_SEMANA_SALIDA', 'SEASON_SALIDA',
        #   'MES_FECHA_TOMA', 'DIA_FECHA_TOMA', 'DIA_SEMANA_FECHA_TOMA', 'SEASON_FECHA_TOMA',
        #   'MES_FECHA_MOD', 'DIA_FECHA_MOD', 'DIA_SEMANA_FECHA_MOD', 'SEASON_FECHA_MOD'
          ]



In [ ]:
# Preparamos los datos
df_modelado = df[variables].copy()

encoder_train_test()
train_test_val()

## **3.1) XGBoost**

In [ ]:

xg, evaluacion = instanciar(evaluacion,
                          'XGBoost - Shap',
                           variables
                          )

## **3.2) CatBoost**

In [ ]:


catB, evaluacion = instanciar(evaluacion,
                            'CatBoost - Shap',
                            variables
                            )

## **3.3) LightGBM**


In [ ]:


lightB, evaluacion = instanciar(evaluacion,
                              'LightGBM - Shap',
                              variables
                              )

## **3.4) Comparativa Curvas ROC**

In [ ]:
modelos = {
      'LightGBM': lightB,
      'XGBoost': xg,
      'CatBoost': catB
  }

comparativa_roc(modelos)

### - Importacia de las variables:

a) Mediante shap values

In [ ]:
df_importantes, variables = funcion_shap(lightB, X_test)

if (('HORA_TOMA_SIN' not in variables) & ('HORA_TOMA_COS' in variables)):
  variables.append('HORA_TOMA_SIN')


df_modelado = df[variables].copy()
train_test_val()
encoder_train_test()

**b) Mediante Permutation Importance**

In [ ]:
# IMPORTANCIA DE VARIABLES:
# ---------------------------------------
df_importantes, variables = importancia_variables(lightB, X_test, y_test)

if (('HORA_TOMA_SIN' not in variables) & ('HORA_TOMA_COS' in variables)):
  variables.append('HORA_TOMA_SIN')

df_modelado = df[variables].copy()
train_test_val()
encoder_train_test()

## **3.3.1) GridSearch LightGBM**

## **a) GridSearch LightGBM Avanzada**

In [ ]:


print("--- INICIANDO OPTIMIZACIÓN AVANZADA DE LIGHTGBM (High Recall Focus) ---")

mejores_hiperparametros = {}

# PREPARACIÓN DE CATEGORÍAS
# Aseguramos que todas las columnas categóricas sean del tipo correcto antes de empezar
variables_categoricas = df_modelado.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

for col in variables_categoricas:

    X_train_t[col] = X_train_t[col].astype(str).astype('category')
    X_test[col] = X_test[col].astype(str).astype('category')
    X_val[col] = X_val[col].astype(str).astype('category')

# CÁLCULO DEL PESO (Mejora sobre is_unbalance)
# Calculamos ratio: (Negativos / Positivos).
# Si queremos priorizar detectar cancelaciones, podemos multiplicar esto por 1.1 o 1.2

ratio_desbalance = (len(y_train_t) - y_train_t.sum()) / y_train_t.sum()
print(f"Ratio calculado para scale_pos_weight: {ratio_desbalance:.2f}")

# Inicializamos el modelo base

lightB = LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=ratio_desbalance,
    verbose=-1
)

# Estrategia de validación cruzada estratificada manteniendo la proporción de cancelaciones

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# -------------------------------------------------------------------------
# PASO A: Estructura del Árbol (Leaf-wise)
# -------------------------------------------------------------------------

print(">>> Paso A: Optimizando num_leaves y max_depth...")

param_grid_a = {
    'num_leaves': [31, 50, 70],        # Un poco más de complejidad permitida
    'max_depth': [-1, 10, 15],
    'min_child_samples': [20, 50, 100] # Evitar overfitting en hojas finales
}

grid_a = GridSearchCV(estimator=lightB,
                      param_grid=param_grid_a,
                      scoring='f1',       # Optimizamos F1 para balancear Precision/Recall
                      cv=cv_strategy,
                      verbose=0)

grid_a.fit(X_train_t, y_train_t)
mejores_hiperparametros.update(grid_a.best_params_)
lightB.set_params(**mejores_hiperparametros)
print(f"Mejores parámetros Paso A: {grid_a.best_params_}")

# -------------------------------------------------------------------------
# PASO B: Aleatoriedad (Robustez)
# -------------------------------------------------------------------------

print(">>> Paso B: Optimizando subsample y colsample...")

param_grid_b = {
    'subsample': [0.7, 0.9],
    'subsample_freq': [1, 5],
    'colsample_bytree': [0.7, 0.9]
}

grid_b = GridSearchCV(estimator=lightB,
                      param_grid=param_grid_b,
                      scoring='f1',
                      cv=cv_strategy,
                      verbose=0)

grid_b.fit(X_train_t, y_train_t)
mejores_hiperparametros.update(grid_b.best_params_)
lightB.set_params(**mejores_hiperparametros)
print(f"Mejores parámetros Paso B: {grid_b.best_params_}")

# -------------------------------------------------------------------------
# PASO C: Regularización (Evitar Falsos Positivos por ruido)
# -------------------------------------------------------------------------

print(">>> Paso C: Optimizando regularización...")

param_grid_c = {
    'reg_alpha': [0, 0.5, 1, 5],      # L1: Ayuda a seleccionar variables
    'reg_lambda': [0, 0.5, 1, 5],     # L2: Suaviza los pesos
    'min_split_gain': [0.0, 0.1]
}

grid_c = GridSearchCV(estimator=lightB,
                      param_grid=param_grid_c,
                      scoring='f1',
                      cv=cv_strategy,
                      verbose=0)

grid_c.fit(X_train_t, y_train_t)
mejores_hiperparametros.update(grid_c.best_params_)
print(f"Mejores parámetros Paso C: {grid_c.best_params_}")

# -------------------------------------------------------------------------
# PASO D: Entrenamiento Final con Early Stopping
# -------------------------------------------------------------------------

print(">>> Paso D: Ajuste final y Early Stopping...")

# Bajamos learning rate y subimos n_estimators para un ajuste fino ("Slow learning")
mejores_hiperparametros['learning_rate'] = 0.01
mejores_hiperparametros['n_estimators'] = 10000

final_model = LGBMClassifier(
    **mejores_hiperparametros,
    objective='binary',
    metric='auc', # LightGBM usará AUC para decidir cuándo parar (es más estable)
    scale_pos_weight=ratio_desbalance,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

callbacks = [
    early_stopping(stopping_rounds=100, verbose=False), # Paciencia de 100 rondas
    log_evaluation(period=0) # Silenciamos el log intermedio
]

final_model.fit(
    X_train_t, y_train_t,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

print(f"Entrenamiento finalizado. Mejor iteración: {final_model.best_iteration_}")

# -------------------------------------------------------------------------
# PASO E: OPTIMIZACIÓN DEL UMBRAL (Threshold Tuning) - CRUCIAL
# -------------------------------------------------------------------------

print("\n>>> Paso E: Buscando el umbral óptimo de decisión...")

# Predecimos probabilidades en VALIDACION
val_proba = final_model.predict_proba(X_val)[:, 1]

# Calculamos curva P-R
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)

# Estrategia: Buscamos el umbral que maximice F1-Score
# OJO: Si te importa MAS el Recall, usa F2-Score (fbeta_score con beta=2)

fscore = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
ix = np.argmax(fscore)
best_thresh = thresholds[ix]

print(f"Umbral óptimo encontrado: {best_thresh:.4f}")
print(f"F-Score esperado en validación: {fscore[ix]:.4f}")
print(f"Recall esperado: {recalls[ix]:.4f} | Precision esperada: {precisions[ix]:.4f}")


# -------------------------------------------------------------------------
# RESULTADO FINAL EN TEST (Usando el nuevo umbral)
# -------------------------------------------------------------------------

proba_train = final_model.predict_proba(X_train_t)[:, 1]
proba_test = final_model.predict_proba(X_test)[:, 1]

# Aplicamos el UMBRAL ÓPTIMO (best_thresh = 0.4779)
pred_train = (proba_train >= best_thresh).astype(int)
pred_test = (proba_test >= best_thresh).astype(int)

# MÉTRICAS TRAIN
auc_train = roc_auc_score(y_train_t, proba_train)
pres_recall_train = average_precision_score(y_train_t, proba_train)
acc_train = accuracy_score(y_train_t, pred_train)
recall_train = recall_score(y_train_t, pred_train)
precision_train = precision_score(y_train_t, pred_train)
f1_score_train = f1_score(y_train_t, pred_train)

# MÉTRICAS TEST
auc_test = roc_auc_score(y_test, proba_test)
pres_recall_test = average_precision_score(y_test, proba_test)
acc_test = accuracy_score(y_test, pred_test)
recall_test = recall_score(y_test, pred_test)
precision_test = precision_score(y_test, pred_test)
f1_score_test = f1_score(y_test, pred_test)

# SOBREAJUSTE
sobreajuste = auc_train - auc_test

# Guardado de resultados
nombre_con_hora = f"LGBM_Optimized_GridS ({hora_actual})"

# Imprimir resultados en consola con el umbral aplicado
print("\n" + "="*40)
print(f"RESULTADOS FINAL EN TEST (Umbral aplicado: {best_thresh:.4f})")
print("="*40)
print(f"AUC (Poder discriminante): {auc_test:.4f}")
print(f"RECALL (Cancelaciones detectadas): {recall_test:.4f}")
print(f"PRECISION (Acierto en predicción): {precision_test:.4f}")
print(f"F1-SCORE (Balance): {f1_score_test:.4f}")
print("-" * 40)

# Reporte de clasificación

print("\nReporte de Clasificación (Set de Test):\n")
print(classification_report(y_test, pred_test))


evaluacion.loc[len(evaluacion)] = {
    'Modelo': nombre_con_hora,
    'auc_train': round(auc_train, 3),
    'auc_test': round(auc_test, 3),
    'precision_recall_train': round(pres_recall_train, 3),
    'precision_recall_test': round(pres_recall_test, 3),
    'accuracy_train': round(acc_train, 3),
    'accuracy_test': round(acc_test, 3),
    'recall_train': round(recall_train, 3),
    'recall_test': round(recall_test, 3),
    'precision_train': round(precision_train, 3),
    'precision_test': round(precision_test, 3),
    'f1_score_train': round(f1_score_train, 3),
    'f1_score_test': round(f1_score_test, 3),
    'sobreajuste': round(sobreajuste,3),
    'variables_utilizadas': variables
}

## **b) Optuna CatBoost**

In [ ]:

def objective(trial):
    # Identificamos las columnas que son texto/categoría
    # Usamos tus variables originales (X_train_t, X_val, etc.)
    variables_categoricas = X_train_t.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    param = {
        'objective': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'thread_count': -1,
        'verbose': False,
        'cat_features': variables_categoricas, # <-- TRATAMIENTO NATIVO

        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 3.0),
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'depth': trial.suggest_int('depth', 4, 8), # CatBoost es sensible a la profundidad
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 2.0, 15.0), # Más alto para evitar sobreajuste
        'random_strength': trial.suggest_float('random_strength', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
    }

    model = CatBoostClassifier(**param)

    # StratifiedKFold para mantener la proporción de cancelaciones
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_t, y_train_t, cv=cv, scoring='roc_auc')
    return scores.mean()

# --- EJECUCIÓN OPTUNA ---
study_cat_native = optuna.create_study(direction='maximize')
study_cat_native.optimize(objective, n_trials=30)

# --- ENTRENAMIENTO FINAL ---
final_params_cat = study_cat_native.best_params.copy()
cat_features = X_train_t.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

final_params_cat.update({
    'objective': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'iterations': 1500,
    'cat_features': cat_features,
    'use_best_model': True
})

final_model_cat = CatBoostClassifier(**final_params_cat)

final_model_cat.fit(
    X_train_t, y_train_t,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=100,
    verbose=100
)

# --- EVALUACIÓN DE RESULTADOS ---
proba_test = final_model.predict_proba(X_test)[:, 1]
preds_test = final_model.predict(X_test)

prob_train = final_model.predict_proba(X_train)[:, 1]
preds_train = final_model.predict(X_train)

# Métricas Train
auc_train = roc_auc_score(y_train, prob_train)
pres_recall_train = average_precision_score(y_train, prob_train)
acc_train = accuracy_score(y_train, preds_train)
recall_train = recall_score(y_train, preds_train)
precision_train = precision_score(y_train, preds_train)
f1_score_train = f1_score(y_train, preds_train)

# Métricas Test

auc_test = roc_auc_score(y_test, proba_test)
pres_recall_test = average_precision_score(y_test, proba_test)
recall_test = recall_score(y_test, preds_test)
acc_test = accuracy_score(y_test, preds_test)
recall_test = recall_score(y_test, preds_test)
precision_test = precision_score(y_test, preds_test)
f1_score_test = f1_score(y_test, preds_test)

# Sobreajuste
sobreajuste = pres_recall_train - pres_recall_test

  #-------------------------------------------------------

  # Probado bajar el umbral y subimos mucho el recall de clase 1. Es decir, de los clientes que realmente cancelaron
  # detectamos un 17 % más pero nos baja mucho la precision (Los que realmente cancelan de los que el modelo predice)

nombre_con_hora = f"Light_OPTUNA_({hora_actual})"

evaluacion.loc[len(evaluacion)] = {'Modelo': nombre_con_hora,
                                    'auc_train': round(auc_train,3),
                                    'auc_test': round(auc_test,3),
                                    'precision_recall_train' : round(pres_recall_train,3),
                                    'precision_recall_test' :  round(pres_recall_test,3),
                                    'accuracy_train' : round(acc_train,3),
                                    'accuracy_test' : round(acc_test,3),
                                    'recall_train' : round(recall_train,3),
                                    'recall_test' : round(recall_test,3),
                                    'precision_train' : round(precision_train,3),
                                    'precision_test': round(precision_test,3),
                                    'f1_score_train': round(f1_score_train,3),
                                    'f1_score_test': round(f1_score_test,3),
                                    'sobreajuste': round(sobreajuste,3),
                                    'variables_utilizadas': variables
                                   }


print()
print(evaluacion.iloc[len(evaluacion)-1])
print()
print(classification_report(y_test, preds_test))
print()

## **c) Optuna LightGBM**

In [ ]:

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'is_unbalance': True,
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'verbose': -1
    }

    variables_categoricas = df_modelado.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    for col in variables_categoricas:
      X_train_t[col] = X_train[col].astype('category')
      X_test[col] = X_test[col].astype('category')
      X_val[col] = X_val[col].astype('category')

    model = LGBMClassifier(**param)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_t, y_train_t, cv=cv, scoring='roc_auc')
    return scores.mean()

# --- EJECUCIÓN OPTUNA ---

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# --- ENTRENAMIENTO FINAL CON MEJORES PARÁMETROS ---

final_params = study.best_params.copy()

# Añadimos los fijos que no estaban en best_params
final_params.update({'objective': 'binary', 'metric': 'auc', 'is_unbalance': True, 'random_state': 42})

final_model = LGBMClassifier(**final_params)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100)
]

final_model.fit(
    X_train_t, y_train_t,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

# --- EVALUACIÓN DE RESULTADOS ---
for col in X_train.select_dtypes(include=['object']).columns:
    X_train[col] = X_train[col].astype('category')

proba_test = final_model.predict_proba(X_test)[:, 1]
preds_test = final_model.predict(X_test)

prob_train = final_model.predict_proba(X_train)[:, 1]
preds_train = final_model.predict(X_train)

# Métricas Train
auc_train = roc_auc_score(y_train, prob_train)
pres_recall_train = average_precision_score(y_train, prob_train)
acc_train = accuracy_score(y_train, preds_train)
recall_train = recall_score(y_train, preds_train)
precision_train = precision_score(y_train, preds_train)
f1_score_train = f1_score(y_train, preds_train)

# Métricas Test

auc_test = roc_auc_score(y_test, proba_test)
pres_recall_test = average_precision_score(y_test, proba_test)
recall_test = recall_score(y_test, preds_test)
acc_test = accuracy_score(y_test, preds_test)
recall_test = recall_score(y_test, preds_test)
precision_test = precision_score(y_test, preds_test)
f1_score_test = f1_score(y_test, preds_test)

# Sobreajuste
sobreajuste = pres_recall_train - pres_recall_test

  #-------------------------------------------------------

  # Probado bajar el umbral y subimos mucho el recall de clase 1. Es decir, de los clientes que realmente cancelaron
  # detectamos un 17 % más pero nos baja mucho la precision (Los que realmente cancelan de los que el modelo predice)

nombre_con_hora = f"Light_OPTUNA_({hora_actual})"

evaluacion.loc[len(evaluacion)] = {'Modelo': nombre_con_hora,
                                    'auc_train': round(auc_train,3),
                                    'auc_test': round(auc_test,3),
                                    'precision_recall_train' : round(pres_recall_train,3),
                                    'precision_recall_test' :  round(pres_recall_test,3),
                                    'accuracy_train' : round(acc_train,3),
                                    'accuracy_test' : round(acc_test,3),
                                    'recall_train' : round(recall_train,3),
                                    'recall_test' : round(recall_test,3),
                                    'precision_train' : round(precision_train,3),
                                    'precision_test': round(precision_test,3),
                                    'f1_score_train': round(f1_score_train,3),
                                    'f1_score_test': round(f1_score_test,3),
                                    'sobreajuste': round(sobreajuste,3),
                                    'variables_utilizadas': variables
                                   }


print()
print(evaluacion.iloc[len(evaluacion)-1])
print()
print(classification_report(y_test, preds_test))
print()

## **d) Optuna XGBoost**

In [ ]:

def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 30), # Más alto para frenar sobreajuste
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 10.0, log=True), # Penalización de splits
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        # Desactivamos lo nativo ya que los datos ya vienen codificados
        'enable_categorical': False,
        'tree_method': 'hist'
    }

    model = xgb.XGBClassifier(**param)

    # Usamos tus sets de datos ya tratados (_D)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_t_D, y_train_t_D, cv=cv, scoring='roc_auc')
    return scores.mean()

# --- EJECUCIÓN OPTUNA ---
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# --- ENTRENAMIENTO FINAL CON MEJORES PARÁMETROS ---
final_params = study.best_params.copy()
final_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
    'enable_categorical': False,
    'tree_method': 'hist'
})

final_model = xgb.XGBClassifier(**final_params)

# Entrenamiento con Early Stopping usando tus sets de validación _D
final_model.fit(
    X_train_t_D, y_train_t_D,
    eval_set=[(X_val_D, y_val_D)],
    verbose=100
)

# --- EVALUACIÓN DE RESULTADOS ---

proba_test = final_model.predict_proba(X_test_D)[:, 1]
preds_test = final_model.predict(X_test_D)

prob_train = final_model.predict_proba(X_train_D)[:, 1]
preds_train = final_model.predict(X_train_D)

# Métricas Train
auc_train = roc_auc_score(y_train_D, prob_train)
pres_recall_train = average_precision_score(y_train_D, prob_train)
acc_train = accuracy_score(y_train_D, preds_train)
recall_train = recall_score(y_train_D, preds_train)
precision_train = precision_score(y_train_D, preds_train)
f1_score_train = f1_score(y_train_D, preds_train)

# Métricas Test

auc_test = roc_auc_score(y_test_D, proba_test)
pres_recall_test = average_precision_score(y_test_D, proba_test)
recall_test = recall_score(y_test_D, preds_test)
acc_test = accuracy_score(y_test_D, preds_test)
recall_test = recall_score(y_test_D, preds_test)
precision_test = precision_score(y_test_D, preds_test)
f1_score_test = f1_score(y_test_D, preds_test)

# Sobreajuste
sobreajuste = pres_recall_train - pres_recall_test

  #-------------------------------------------------------

  # Probado bajar el umbral y subimos mucho el recall de clase 1. Es decir, de los clientes que realmente cancelaron
  # detectamos un 17 % más pero nos baja mucho la precision (Los que realmente cancelan de los que el modelo predice)

nombre_con_hora = f"XGBoost_OPTUNA_({hora_actual})"

evaluacion.loc[len(evaluacion)] = {'Modelo': nombre_con_hora,
                                    'auc_train': round(auc_train,3),
                                    'auc_test': round(auc_test,3),
                                    'precision_recall_train' : round(pres_recall_train,3),
                                    'precision_recall_test' :  round(pres_recall_test,3),
                                    'accuracy_train' : round(acc_train,3),
                                    'accuracy_test' : round(acc_test,3),
                                    'recall_train' : round(recall_train,3),
                                    'recall_test' : round(recall_test,3),
                                    'precision_train' : round(precision_train,3),
                                    'precision_test': round(precision_test,3),
                                    'f1_score_train': round(f1_score_train,3),
                                    'f1_score_test': round(f1_score_test,3),
                                    'sobreajuste': round(sobreajuste,3),
                                    'variables_utilizadas': variables
                                   }


print()
print(evaluacion.iloc[len(evaluacion)-1])
print()
print(classification_report(y_test_D, preds_test))
print()

# **4) Graficos y salvado de resultados**

In [ ]:
evaluacion

In [ ]:
evaluacion.sort_index()

In [ ]:
# ---- PREPARO RUTAS PARA GUARDAR

ruta_base = "/content/drive/MyDrive/TFM_Aurea_Team/02_notebooks/Modelado/grafica_david/"
ruta_dia = os.path.join(ruta_base, fecha_folder)
ruta_final = os.path.join(ruta_dia, hora_actual)
os.makedirs(ruta_final, exist_ok=True)

In [ ]:
# ---- GUARDAR X_TEST

nombre_csv = f"X_test_{hora_actual}.csv"
ruta_csv = os.path.join(ruta_final, nombre_csv)
X_test.to_csv(ruta_csv, index=False, sep=';')

In [ ]:
# ---- GUARDAR Y_TEST
nombre_csv = f"y_test_{hora_actual}.csv"
ruta_csv = os.path.join(ruta_final, nombre_csv)
y_test.to_csv(ruta_csv, index=False, sep=';')

In [ ]:
# --- GUARDAR DATAFRAME (CSV) ---

nombre_csv = f"comparativa_modelos_variables_OPTUNA_{hora_actual}.csv"
ruta_csv = os.path.join(ruta_final, nombre_csv)
evaluacion.to_csv(ruta_csv, index=False, sep=';')
print(f"✅ CSV guardado en : {ruta_csv}")

In [ ]:

#nombre_grafica = f"grafico_comparativa_azul_{hora_actual}.png"
#ruta_grafica = os.path.join(ruta_final, nombre_grafica)

rendimiento(proba_test, preds_test, y_test, ruta_final)

In [ ]:
# Transformo el dataframe en formato largo

largo = evaluacion.melt(id_vars='Modelo', var_name='Metrica', value_name='Valor')

In [ ]:
#--- GRAFICO ---

# Configuración de modelos y colores

modelos_unicos = sorted(evaluacion['Modelo'].unique())
image_palette = sns.color_palette("deep", len(modelos_unicos))
color_map = dict(zip(modelos_unicos, image_palette))

# Filtrar solo métricas relevantes

all_metrics = sorted(largo['Metrica'].unique())
metric_order = sorted(m for m in all_metrics if ('_test' in m.lower()))# or ('_train' in m.lower())])

# Configuración del gráfico

fig, ax = plt.subplots(figsize=(15, 12))
background_color = '#FDFBFB'
fig.patch.set_facecolor(background_color)
ax.set_facecolor(background_color)

factor_espaciado = 1.15
y_positions = np.arange(len(metric_order)) * factor_espaciado
height_group = 0.90
bar_height = height_group / len(modelos_unicos)

# Dibujado de barras

for i, metrica in enumerate(metric_order):
    df_metrica = largo[largo['Metrica'] == metrica].sort_values(by='Valor', ascending=False).reset_index()

    y_center = y_positions[i]
    y_pos_bars = y_center - (height_group/2) + (np.arange(len(df_metrica)) * bar_height) + (bar_height/2)

    bars = ax.barh(y_pos_bars, df_metrica['Valor'], height=bar_height,
                    color=[color_map[m] for m in df_metrica['Modelo']],
                    edgecolor='none')

    for bar in bars:
        width = bar.get_width()
        if not np.isnan(width) and width > 0.04:
            ax.text(width/2, bar.get_y() + bar.get_height()/2, f'{width*100:.0f}%',
                    va='center', ha='center', color='black', fontweight='bold', fontsize=9)

# Ejes y Estética

ax.set_yticks(y_positions)
ax.set_yticklabels(metric_order, fontsize=11, fontweight='bold', color='#2C3E50')
ax.invert_yaxis()
ax.spines['left'].set_visible(True); ax.spines['left'].set_color('#2C3E50')
ax.spines['bottom'].set_visible(True); ax.spines['bottom'].set_color('#2C3E50')
ax.xaxis.grid(True, linestyle='--', alpha=0.3, color='#2C3E50')
ax.set_axisbelow(True)

ax.set_title('COMPARATIVA DE MÉTRICAS', fontsize=20, pad=35, fontweight='bold', color='#2C3E50')
ax.set_xlabel('Valor (0-1)', fontsize=12, fontweight='bold', color='#2C3E50')

# --- LEYENDA---
legend_elements = [Line2D([0], [0], color=color_map[m], lw=10, label=m) for m in modelos_unicos]
leg = ax.legend(handles=legend_elements, title='Modelos', bbox_to_anchor=(1.02, 1),
                loc='upper left', frameon=False, labelcolor='#2C3E50')

# --- SOBREAJUSTE ---

# Construimos el texto dinámicamente
texto_sobreajuste = "SOBREAJUSTE (precision- recall):n" + "-"*25 + "\n"
for modelo in modelos_unicos:

    # Obtenemos el valor del dataframe comparativo
    val_overfit = evaluacion[evaluacion['Modelo'] == modelo]['sobreajuste'].values[0]

    # Definimos color del texto según gravedad (Opcional, ayuda visual)
    alerta = "⚠️" if val_overfit > 0.10 else "✅"
    texto_sobreajuste += f"{modelo}: {val_overfit:.3f} {alerta}\n"

# Lo situamos en y=0.75 (Debajo de la leyenda que está en y=1.0)
plt.text(1.03, 0.70, texto_sobreajuste,
         transform=ax.transAxes,
         fontsize=10, fontfamily='monospace', # Monospace para que alineen los números
         verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff3cd', edgecolor='#e0a800', alpha=0.5), # Color amarillento de alerta
         color='#2C3E50')

# --- RECUADRO DE VARIABLES ---

# Ajustado a y=0.35 para dejar espacio al nuevo cuadro
texto_variables = ", ".join(variables) if 'variables' in locals() else "Variables no definidas"
texto_envolvente = textwrap.fill(f"Variables usadas:\n{texto_variables}", width=40)

plt.text(1.03, 0.35, texto_envolvente,
         transform=ax.transAxes,
         fontsize=10,
         verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FDFBFB', edgecolor='#2C3E50', alpha=0.3),
         color='#2C3E50')


nombre_grafica = f"grafico_comparativa_{hora_actual}.png"
ruta_grafica = os.path.join(ruta_final, nombre_grafica)
plt.savefig(ruta_grafica, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())

plt.tight_layout()
plt.show()

# **5) Guardar Modelo**

In [ ]:
# -------------------------------------------------------------------------
# GUARDADO MODELO EN DRIVE
# -------------------------------------------------------------------------
#

project_path = '/content/drive/MyDrive/TFM_Aurea_Team/'
models_folder = os.path.join(project_path, "04_models")
experiment_dir = os.path.join(models_folder, f"lightgbm_FINAL{hora_actual}")
os.makedirs(experiment_dir, exist_ok=True)

model_filename = "lightgbm.joblib"
model_path = os.path.join(experiment_dir, model_filename)
joblib.dump(final_model, model_path)

# Me aseguro que best_iteration existe (si no, poner 0 o n_estimators)
best_iter = getattr(final_model, 'best_iteration_', 0)
if best_iter is None: best_iter = 0

metadata = {
    "experiment_id": hora_actual,
    "model_type": "LightGBM",
    "model_file": model_filename,
    "parametros_finales": final_params,
    "metricas_test": {
        "accuracy": float(acc_test),
        "auc_roc": float(auc_test),
        "average_precision_score": float(pres_recall_test),
        "recall_score": float(recall_test),
        "precision_score": float(precision_test),
        "f1_score": float(f1_score_test),
        "best_iteration": int(best_iter)
    },
    "classification_report": classification_report(y_test, preds_test, output_dict=True),
    "variables_usadas": variables if isinstance(variables, list) else list(variables)
}

# 4. Guardar JSON de métricas
metrics_path = os.path.join(experiment_dir, "metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ [OK] Modelo y métricas guardados con éxito.")
print(f"📁 Carpeta: {experiment_dir}")

In [ ]:
# Guardar variables

import csv


# 1. Definimos el nombre del archivo
variables_filename = 'variables_utilizadas.csv'

variables_path = os.path.join(experiment_dir, variables_filename)

# 2. Guardamos la lista
with open(variables_path, 'w', newline='') as f:
    writer = csv.writer(f)
    for val in variables:
        writer.writerow([val])

print(f"✅ Lista de variables guardada en: {variables_path}")

In [ ]:

ruta = '/content/drive/MyDrive/TFM_Aurea_Team/04_models/lightgbm_FINAL20260206_142533'
path_a_cargar = os.path.join(experiment_dir, 'variables_utilizadas.csv')

# Cargar
if os.path.exists(path_a_cargar):
    with open(path_a_cargar, 'r') as f:
        reader = csv.reader(f)
        # Cargamos asegurando que no queden filas vacías
        lista_cargada = [row[0] for row in reader if row]

    print(f"✅ Variables cargadas con éxito ({len(lista_cargada)} variables).")
    print(lista_cargada)
else:
    print(f"❌ Error: No se encontró el archivo en {path_a_cargar}")



In [ ]:

ruta_experimento = '/content/drive/MyDrive/TFM_Aurea_Team/04_models/lightgbm_FINAL20260206_142533'

# Cargar el modelo (.joblib)
model_path = os.path.join(ruta_experimento, "lightgbm.joblib")
loaded_model = joblib.load(model_path)

# Cargar las métricas y metadatos (.json)
metrics_path = os.path.join(ruta_experimento, "metrics.json")
with open(metrics_path, 'r') as f:
    loaded_metadata = json.load(f)

# --- VERIFICACIÓN ---
print(f"✅ Modelo {loaded_metadata['model_type']} cargado correctamente.")
print(f"📈 AUC en Test: {loaded_metadata['metricas_test']['auc_roc']:.4f}")
print(f"📋 Variables utilizadas: {len(loaded_metadata['variables_usadas'])}")